# Related Papers:
- https://aclanthology.org/2024.acl-short.2.pdf
- https://aclanthology.org/2024.emnlp-main.171.pdf
- https://arxiv.org/pdf/2408.09174v1
- https://aclanthology.org/2025.findings-emnlp.638.pdf
- https://aclanthology.org/2025.emnlp-main.363.pdf

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_jsonl(file_name, records):
    path = os.path.join(REPO_PATH, "data_files", f"{file_name}.jsonl")
    with open(path, "w", encoding="utf-8") as out_file:
        for record in records:
            out_file.write(json.dumps(record, ensure_ascii=False) + "\n")

# Data Creation:

## Harvest Wikipedia pages with tables:

In [91]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester()
# Run Phase 1 - Harvesting titles based on keywords
harvester.run_complete_harvest()
# Run Phase 2 - Filtering pages for tables
await harvester.filter_pages_async()

Harvesting Pages By Keywords:   0%|          | 0/233 [00:00<?, ?keyword/s]


TOTAL UNIQUE TITLES COLLECTED: 167913



Method,Keyword,Count
Prefix,List of cities,1537
Prefix,List of countries,1278
Prefix,List of towns,501
Prefix,List of villages,483
Prefix,List of districts,375
Prefix,List of municipalities,1021
Prefix,List of communes,194
Prefix,List of counties,259
Prefix,List of provinces,84
Prefix,List of territories,22


Filtering Pages with Tables:   0%|          | 0/167913 [00:00<?, ?page/s]


TOTAL PAGES WITH TABLES COLLECTED: 62306



In [93]:
save_json("table_titles", harvester.valid_titles)

In [174]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

## Parse and filter Wikipedia tables:

In [271]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(table_titles)
validated_tables = load_jsonl(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
cleaned_data = processor.process()

Pre-processing: cluster similar pages and discard similarities ---> 6426 pages discarded.


Extracting Tables:   0%|          | 0/55880 [00:00<?, ?Page/s]

Starting Extraction: 55880 titles | Async Limit: 20 | CPU Workers: 19
TOTAL PAGES WITH VALID TABLES: 32076 OUT OF 55880 (57.4%)


Filtering & Cleaning:   0%|          | 0/94101 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      94101
Dropped (Bad Headers):     8435
Dropped (Cell too long):   2841
Dropped (Repeat Rows):     1783
Short tables found:        39837
Unique source pages:       16940
Final sampled (1/page):    16940
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

,Rank,Player,Career,Caps,Goals
0,1,Christine Sinclair,2000–2023,331,190
1,2,Sophie Schmidt,2005–2023,226,20
2,3,Diana Matheson,2003–2021,206,19
3,4,Desiree Scott,2010–2025,188,0
4,5,Rhian Wilkinson,2003–2017,181,7
5,6,Kadeisha Buchanan,2013–present,154,6
6,7,Ashley Lawrence,2013–present,151,8
7,7,Jessie Fleming,2013–present,151,20
8,9,Brittany Baxter,2002–2014,132,5
9,9,Andrea Neil,1991–2007,132,24



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Manchester City F.C. 3–2 Queens Park Rangers F.C. > Match > Statistics
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Statistic,Manchester City,Queens Park Rangers
0,Goals scored,3,2
1,Possession,81.3%,18.7%
2,Shots on target,15,3
3,Shots off target,10,0
4,Blocked shots,19,0
5,Corner kicks,19,0
6,Fouls,4,7
7,Tackles,15,22
8,Offsides,1,1
9,Yellow cards,2,1



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Swinburne University of Technology > Notable people > Academics and staff
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Member,Executive team,Period
0,Pascale Quester,Vice-Chancellor,2020–present
1,Chris Pilgrim,Pro Vice-Chancellor (Academic),
2,Tara Magdalinski,Pro Vice-Chancellor (Education and Quality),
3,Sarah Maddison,Pro Vice-Chancellor (Academic Innovation and C...,
4,Bronwyn Fox,Deputy Vice-Chancellor (Research and Enterprise),
5,Beth Webster,Pro Vice-Chancellor (Research Policy and Impact),
6,Qing-Lon Han,Pro Vice-Chancellor (Research Quality),
7,Alan Kin-Tak Lau,Pro Vice-Chancellor (International Research De...,
8,Blair Kuys,Dean of the School of Design and Architecture,
9,Matthew Bailes,Director of the Data Science Research Institut...,


## Create Potential Entity Groups:

In [296]:
list_of_cleaned_data = [a for a in cleaned_data if a['table_name'].lower().startswith('list of')]
len(list_of_cleaned_data)

5690

In [297]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(list_of_cleaned_data, 1000))
await group_constructor.get_groups()

🚀 Grouping 1000 tables using Entity-Focus logic...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1000 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════════════════════════
                               📊 GROUPING REPORT                                
════════════════════════════════════════════════════════════════════════════════
  ✅ Valid Groups:   933
  ❌ Invalid Tables: 67
════════════════════════════════════════════════════════════════════════════════

✨ SAMPLE VALID NAMES:
  → Towns on the Isle of Wight
  → Senior Staff Officers of the Indian Air Force
  → Railway stations in India starting with O

────────────────────────────────────────────────────────────────────────────────
                     🕵️ REJECTION DEBUGGER (Random Sample)                      
────────────────────────────────────────────────────────────────────────────────

[TABLE]: List of founders of religious traditions > Bibliography
[REASON]: The table contains categories of religious traditions rather than unique instances of entities. It does not meet the criteria for a coherent group of entities

In [310]:
stst = []
for sample in random.sample(group_constructor.valid_tables, 100):
    try:
        stst.append(list(sample['Table data'][0].keys()).index(sample['Entity column']))
        print(sample['Group name'])
        print(sample['Entity column'])
        for row in sample['Table data'][:3]:
            print(row)
        print("=" * 100)
        # display(pd.DataFrame(sample['Table data']))
    except:
        pass
        stst.append('Failed')
# print(Counter(stst))

Grammy Awards received by Carly Simon
Award
{'Year': 1974, 'Work': 'No Secrets', 'Award': 'Best Engineered Recording', 'Recipient': 'Robin Geoffrey Cable and Bill Schnee', 'Result': 'Nominated'}
{'Year': 1976, 'Work': 'Playing Possum', 'Award': 'Best Album Package', 'Recipient': 'Gene Christensen A', 'Result': 'Nominated'}
{'Year': 1978, 'Work': '" Nobody Does It Better "', 'Award': 'Song of the Year', 'Recipient': 'Marvin Hamlisch and Carole Bayer Sager', 'Result': 'Nominated'}
Massacres in the Ottoman Empire before 1914
Name
{'Name': 'Massacres during the Greek War of Independence', 'Date': '1821–1829', 'Location': 'Ottoman Empire', 'Deaths': 'Unknown', 'Responsible Party': 'Ottoman government and Greek rebels', 'Victims': 'Greeks , Turks, Albanians and Jews', 'Notes': 'Massacres were committed by both sides during the conflict.'}
{'Name': 'Massacres of Badr Khan', 'Date': '1843-1846', 'Location': 'Hakkari', 'Deaths': '4,000-7,000', 'Responsible Party': 'Kurdish Emirs of Bhutan, Badr

In [663]:
for sample in random.sample(group_constructor.valid_tables, 25):
    print(f"Group name: {sample['Group name']}")
    print(f"Wikipedia page: {sample['Wikipedia link']}")
    print(f"Entity column: {sample['Entity column']}")
    display(pd.DataFrame(sample['Table data']))
    print()
    print()

Group name: Artists with multiple nominations and awards at the Brit Awards 2006
Wikipedia page: https://en.wikipedia.org/wiki/Brit_Awards_2006#Multiple_nominations_and_awards
Entity column: Artist


,Awards,Artist
0,3,Kaiser Chiefs
1,2 (3),Coldplay
2,2 (3),Green Day
3,2 (3),James Blunt




Group name: Awards and nominations received by Sam Smith at the iHeartRadio Music Awards
Wikipedia page: https://en.wikipedia.org/wiki/List_of_awards_and_nominations_received_by_Sam_Smith#iHeartRadio_Music_Awards
Entity column: Nominee / work


,Year,Nominee / work,Award,Result
0,2015,Sam Smith,Artist of the Year,Nominated
1,2015,Sam Smith,Best New Artist,Won
2,2015,"""Stay with Me""",Song of the Year,Nominated
3,2015,"""Stay with Me""",Best Lyrics,Nominated
4,2016,Sam Smith,Male Artist of the Year,Nominated
5,2016,In the Lonely Hour,Album of the Year,Nominated
6,2016,"""Writing's on the Wall""",Best Song from a Movie,Nominated
7,2016,""" Hotline Bling """,Best Cover Song,Nominated
8,2020,"""Dancing With A Stranger"" (with Normani)",Best Collaboration,Nominated
9,2020,"""Dancing With A Stranger"" (with Normani)",Best Music Video,Nominated




Group name: 1965 Tennessee Volunteers football schedule
Wikipedia page: https://en.wikipedia.org/wiki/1965_Tennessee_Volunteers_football_team#Schedule
Entity column: Opponent


...

,Race / Ethnicity ( NH = Non-Hispanic ),Pop 2000,Pop 2010,Pop 2020,% 2000,% 2010,% 2020
0,White (NH),122708,133790,128348,82.79%,79.24%,73.25%
1,Black or African American (NH),15526,17844,17549,10.48%,10.57%,10.02%
2,Native American or Alaska Native (NH),1096,1033,795,0.74%,0.61%,0.45%
3,Asian (NH),2534,3298,4068,1.71%,1.95%,2.32%
4,Native Hawaiian or Pacific Islander alone (NH),111,140,194,0.07%,0.08%,0.11%
5,Other race alone (NH),164,223,720,0.11%,0.13%,0.41%
6,Mixed race or Multiracial (NH),2487,4417,9696,1.68%,2.62%,5.53%
7,Hispanic or Latino,3591,8107,13846,2.42%,4.80%,7.90%




Group name: Guns N' Roses touring members
Wikipedia page: https://en.wikipedia.org/wiki/List_of_Guns_N'_Roses_members#Members
Entity column: Name


,Name,Years active,Instruments,Details
0,Fred Coury,1987–1988,drums,Coury substituted for Steven Adler between lat...
1,"Kid ""Haggis"" Chaos (Stephen Harris)",1988,bass,Harris replaced Duff McKagan for a show on May...
2,Don Henley,1989,drums backing vocals,Henley replaced Steven Adler for the 1989 Amer...
3,Teddy Andreadis,1991–1993,keyboards percussion harmonica backing vocals,Additional members of the Use Your Illusion To...
4,Roberta Freeman,1991–1993,backing vocals tambourine,Additional members of the Use Your Illusion To...
5,Tracey Amos,1991–1993,backing vocals tambourine,Additional members of the Use Your Illusion To...
6,Diane Jones,1991–1993,backing vocals tambourine,Additional members of the Use Your Illusion To...
7,Cece Worrall,1991–1993,saxophones,Additional members of the Use Your Illusion To...
8,Lisa Maxwell,1991–1993,saxophones,Additional members of the Use Your Illusion To...
9,Anne King,1991–1993,trumpet,Additional members of the Use Your Illusion To...




Group name: Racial and ethnic composition of Rapids City, Illinois
Wikipedia page: https://en.wikipedia.org/wiki/Rapids_City,_Illinois#Demographics
Entity column: Race / Ethnicity ( NH = Non-Hispanic )


,Race / Ethnicity ( NH = Non-Hispanic ),Pop 2000,Pop 2010,Pop 2020,% 2000,% 2010,% 2020
0,White alone (NH),924,922,889,96.96%,96.14%,92.22%
1,Black or African American alone (NH),0,2,8,0.00%,0.21%,0.83%
2,Native American or Alaska Native alone (NH),0,1,1,0.00%,0.10%,0.10%
3,Asian alone (NH),1,3,1,0.10%,0.31%,0.10%
4,Native Hawaiian or Pacific Islander alone (NH),1,0,0,0.10%,0.00%,0.00%
5,Other race alone (NH),1,0,1,0.10%,0.00%,0.10%
6,Mixed race or Multiracial (NH),6,10,27,0.63%,1.04%,2.80%
7,Hispanic or Latino (any race),20,21,37,2.10%,2.19%,3.84%


## Create FanOut-style Questions out of the Entity Groups:

### Rule-Based Strategy Candidate Selection:

In [311]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(group_constructor.valid_tables)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/933 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/933 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     897
❌ Groups Discarded:  36
📈 Success Rate:      96.1%
-------------------------------------------------------
Strategy Distribution (Top 4 per table):
  - UNIQUE_DIVERSITY          : 657
  - COUNT_TOTAL               : 473
  - EXTREME_TEMPORAL          : 472
  - FILTER_COUNT_TEMPORAL     : 354
  - FREQUENCY_MODE            : 266
  - CATEGORICAL_FILTER_LOOKUP : 235
  - EXTREME_NUMERICAL         : 91
  - AGGREGATION               : 84
  - TOP_K_RANKING             : 83
  - COMPARATIVE_DIFFERENCE    : 57
  - FILTER_COUNT_NUMERICAL    : 34
-------------------------------------------------------
POSSIBLE STRATEGIES  | GROUP COUNT  | PERCENTAGE
-------------------------------------------------------
#1                   | 79           |       8.8%
#2                   | 182          |      20.3%
#3 

In [316]:
selector.debug_strategy_samples("FREQUENCY_MODE")

Found 266 total tables with this strategy. Showing 5 random samples.



🔗 Source: https://en.wikipedia.org/wiki/List_of_motor_vehicle_awards#People


,Award,Sponsor
0,Automotive Executive of the Year Award,DNV GL - Business Assurance
1,Automotive Hall of Fame,MotorCities National Heritage Area
2,Car Designer of the Century,Global Automotive Elections Foundation
3,Car Engineer of the Century,Global Automotive Elections Foundation
4,Car Entrepreneur of the Century,Global Automotive Elections Foundation
5,Louis Schwitzer Award,Society of Automotive Engineers
6,Time Magazine Quality Dealer Award,"Time , Ally Financial"


🔗 Source: https://en.wikipedia.org/wiki/List_of_new_members_of_the_109th_United_States_Congress#Senate


,Senator,Prior background
0,Ken Salazar (D),Attorney General of Colorado
1,Mel Martinez (R),U.S. Secretary of Housing and Urban Developmen...
2,Johnny Isakson (R),U.S. House of Representatives Georgia Senate ...
3,Barack Obama (D),Illinois Senate
4,David Vitter (R),U.S. House of Representatives Louisiana House...
5,Richard Burr (R),U.S. House of Representatives
6,Tom Coburn (R),U.S. House of Representatives
7,Jim DeMint (R),U.S. House of Representatives
8,John Thune (R),U.S. House of Representatives


...

,Nominated work,Category
0,August: Osage County,Outstanding Cast in a Motion Picture
1,12 Years a Slave,Outstanding Cast in a Motion Picture
2,The Imitation Game,Outstanding Cast in a Motion Picture
3,The Imitation Game,Outstanding Male Actor in a Leading Role
4,Sherlock,Outstanding Male Actor in a Miniseries or Movie
5,Sherlock,Outstanding Male Actor in a Miniseries or Movie
6,The Power of the Dog,Outstanding Male Actor in a Leading Role


🔗 Source: https://en.wikipedia.org/wiki/List_of_power_stations_in_Idaho#Renewable_power_stations


,Name,Primary fuel
0,Bannock County Landfill Gas,Landfill gas
1,Cargill Dry Creek Biofactory,Biogas
2,Clearwater Paper Biomass,Wood/wood waste
3,Fighting Creek Landfill Gas,Landfill gas
4,Hidden Hollow Energy,Landfill gas
5,Milner Butte LFGE,Landfill gas
6,Plummer Cogen,Wood/wood waste
7,Rock Creek Dairy,Biogas
8,Simplot Don Plant,Industrial waste heat
9,Tamarack Energy,Wood/wood waste


🔗 Source: https://en.wikipedia.org/wiki/List_of_Dungeons_&_Dragons_video_games#List_of_games


,Title,Setting
0,Dungeons & Dragons Computer Fantasy Game,Generic
1,Advanced Dungeons & Dragons: Cloudy Mountain,Generic
2,Advanced Dungeons & Dragons: Treasure of Tarmin,Generic
3,Spelljammer: Pirates of Realmspace,Spelljammer
4,Stronghold,Generic
5,Al-Qadim: The Genie's Curse,Al-Qadim
6,Slayer,Generic
7,DeathKeep,Generic
8,Birthright: The Gorgon's Alliance,Birthright
9,Dungeons & Dragons Tactics,Generic


### Model-Based Question Generation:

In [312]:
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

generator = QuestionGenerator(random.sample(selector.updated_data, 100),
                              strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await generator.generate_questions()

🚀 Starting Question Generation for 100 tables...
--- Round 1: Processing 100 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/100 [00:00<?, ?it/s]

--- Round 2: Processing 1 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1 [00:00<?, ?it/s]


═════════════════════════════════════════════════════════════════
                  📊 QUESTION GENERATION SUMMARY                  
═════════════════════════════════════════════════════════════════
Initial Tables:      100
Successfully Gen:    100
Exhausted Strats:    0
Success Rate:        100.0%
-----------------------------------------------------------------
STRATEGY                     | SUCCESS  | TRIED    | RATE
-----------------------------------------------------------------
COUNT_TOTAL                  | 16       | 16       | 100%
UNIQUE_DIVERSITY             | 15       | 15       | 100%
EXTREME_TEMPORAL             | 14       | 14       | 100%
FILTER_COUNT_TEMPORAL        | 13       | 13       | 100%
FREQUENCY_MODE               | 13       | 14       |  93%
CATEGORICAL_FILTER_LOOKUP    | 13       | 13       | 100%
AGGREGATION                  | 4        | 4        | 100%
EXTREME_NUMERICAL            | 3        | 3        | 100%
TOP_K_RANKING                | 3        | 3   

,Award,Sponsor
0,Automotive Executive of the Year Award,DNV GL - Business Assurance
1,Automotive Hall of Fame,MotorCities National Heritage Area
2,Car Designer of the Century,Global Automotive Elections Foundation
3,Car Engineer of the Century,Global Automotive Elections Foundation
4,Car Entrepreneur of the Century,Global Automotive Elections Foundation
5,Louis Schwitzer Award,Society of Automotive Engineers
6,Time Magazine Quality Dealer Award,"Time , Ally Financial"


-----------------------------------------------------------------


## Question Validation:

### Model-Based Question Validator (Gemini 2.5 Flash):

In [313]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(generator.valid_results, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/91 [00:00<?, ?it/s]

Network
Lifetime
NBC
NBC
ESPN
NBC


Airline,2024
United Airlines,311
American Airlines,293
Delta Air Lines,288
Southwest Airlines,178
Air Canada,104
JetBlue,66
Alaska Airlines,64
Spirit Airlines,53
Frontier Airlines,40


Name,Years active
Steve Harris,1975–present
Dave Murray,1976–1977 1978–present
Adrian Smith,1980–1990 1999–present
Bruce Dickinson,1981–1993 1999–present
Nicko McBrain,1982–present (inactive from touring as of 2024)
Janick Gers,1990–present


Title,Month
Pathaan,January
Chhaava,February
The Kashmir Files,March
Kisi Ka Bhai Kisi Ki Jaan,April
Yeh Jawaani Hai Deewani,May
Sanju,June
Bajrangi Bhaijaan,July
Stree 2,August
Jawan,September
Secret Superstar,October


Name,ITIS TSN
Bombus alpinus,714857
Bombus balteatus,714786
Bombus hyperboreus,714811
Bombus kirbiellus,1128121
Bombus kluanensis,1129722
High Arctic Bumble Bee,1129726
Bombus neoboreus,714825
Bombus polaris,714831
Bombus pyrrhopygus,1129727


Title,Year
Cléopâtre,1910
Cleopatra,1912
Antony and Cleopatra,1913
Cleopatra,1917


Coach,Coach of the Year
Bruce Arena,"4 (1997, 2009, 2011, 2021)"
Bob Bradley,"3 (1998, 2006, 2019)"
Sigi Schmid,"2 (1999, 2008)"
Frank Yallop,"2 (2001, 2012)"
Jim Curtin,"2 (2020, 2022)"
Dominic Kinnear,1 (2005)
Greg Vanney,1 (2017)
Caleb Porter,1 (2013)
Thomas Rongen,1 (1996)
Brian Schmetzer,—


### Manual Validation:

In [318]:
tagger = ManualTagger(validator.approved_data, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
tagger.display() # Render the output area

📦 Progress Loaded: 14 samples already tagged in 'tagger_progress.json'.


Output()

# Tests:

## Harvest Wikipedia pages with tables:

1. **The Discovery Phase (The Wide Net)**
   - Input: 22 Subject Groups (e.g., Infrastructure, Sports).
   - Action: We hit the Wikipedia API with two distinct patterns:
     - Prefix Mining: Grabbing every page starting with "Entity in/of..."
     - Search Mining: Finding every page with "List of [Entity]" in the title.
     - Goal: Maximum recall. We want to find everything that might be a list.
2. **The Validation Phase (The Filter)**
   - Action: We open every page found in Phase 1 (asynchronously for speed).
   - Logic: We look for the "DNA" of a table: the {| or wikitable tags.
   - Outcome: We discard the "stub" articles or prose-only pages that don't actually contain structured data.
3. **The Clustering Phase (The De-duplicator)**
   - Problem: Wikipedia often has "List of 2023 Films" and "List of films of 2023." These are duplicates.
   - Action: Using TF-IDF to cluster similar page titles.
   - If two titles are >80% similar, we draw a line (edge) between them.
   - Pick a representative from each cluster to continue.

In [97]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiMiner

miner = WikiMiner()
await miner.extract()

Mining Pages:   0%|          | 0/311 [00:00<?, ?entity/s]

Filtering Pages with Tables:   0%|          | 0/87615 [00:00<?, ?page/s]

Computing Similarities for 43421 Titles:   0%|          | 0/43421 [00:00<?, ?titles/s]

Picking Representatives from 38899 Clusters:   0%|          | 0/38899 [00:00<?, ?cluster/s]

## Parse and filter valid Wikipedia tables:

1. **Geometric Fix** - Rebuilds the "hidden grid" of the table. It maps merged cells (rowspans) to their correct spots so the data doesn't shift.
2. **Context Grab** - Captures the Section Path (e.g., Earthquakes > 1954) and the Intro Sentence above the table to explain what the data means.
3. **Trash Filter** - Deletes "Legend" tables, empty tables, and tiny snippets. If it's less than 4 rows or 50% empty, it’s gone.
4. **Variety Control** - "Detects ""Clones"" (like 12 identical monthly tables). It keeps a maximum of 4 different tables per page to ensure the dataset isn't boring.

In [130]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/candidate_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(miner.valid_titles)

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/candidate_tables.jsonl"))
cleaned_data = processor.process()

Extracting Tables:   0%|          | 0/38899 [00:00<?, ?Page/s]

Starting Extraction: 38899 titles | Async Limit: 20 | CPU Workers: 7


Filtering & Cleaning:   0%|          | 0/65565 [00:00<?, ?it/s]

Legend / Key Filter,496
Header Parsing Failures,"4,183"
Cell Content Too Long,"1,852"
Malformed Row Structures,"2,006"
Intra-Page Duplicates Removed,"15,895"


In [119]:
processor.display_sample()

,Rank,Region,Numbers (in thousands),Percentage of world total numbers,As percentage of total adult population
0,-,World,62489,100.0,1.1
1,1,Northern America,26778,41.9,9.5
2,2,Europe,16696,26.7,2.8
3,3,Asia-Pacific,10755,17.2,0.8
4,4,China,6190,9.9,0.6
5,5,Latin America,915,1.5,0.2
6,6,India,796,1.3,0.1
7,7,Africa,352,0.6,0.1


## Create Potential Entity Groups:

In [192]:
raw_dat = load_jsonl(os.path.join(REPO_PATH, "data_files/candidate_tables.jsonl"))
len(raw_dat)

65565

In [196]:
raw_dat[0]['table_name']['header_chain']

'List of legislatures by country > List of legislatures > Supranational legislatures'

In [221]:
from collections import defaultdict

def long_tables_no_short_page(raw_dat):
    pages = defaultdict(list)

    for s in raw_dat:
        page = s['table_name']['header_chain'].split(" > ")[0]
        pages[page].append(s)

    return [
        {'title': t['table_name']['header_chain'], 'cols': list(t['rows'][0].keys())}
        for tables in pages.values()
        if all(len(t['rows']) > 12 for t in tables)
        for t in tables
    ]
long_tables = long_tables_no_short_page(raw_dat)
len(long_tables)

14765

In [229]:
SUPERLATIVE_WORDS = ["tallest", "highest", "largest", "biggest", "longest", "shortest", "fastest", "slowest", "oldest", "youngest", "lowest", "richest", "poorest", "deadliest", "busiest", 
                     "costliest", "hottest", "brightest", "strongest", "cleanest", "closest", "narrowest", "earliest", "latest", "greatest", "fewest", "best", "top", "most", "grossing"]
processed = set()
is_ranked = False


for table in long_tables:

    header_chain = table.get('title', {}).lower()
    first_col_name = table.get('cols', [])[0]

    # Match your logic: "Rank" in first col OR superlatives in header
    is_ranked = 'rank' in first_col_name.lower() or any(x in header_chain for x in SUPERLATIVE_WORDS)

    if is_ranked:
        processed.add(header_chain.split(" > ")[0])
        is_ranked = False
print(len(processed))
for p in processed:
    print(f"--> {p}")

912
--> list of cities in brandenburg by population
--> list of cities in kanagawa prefecture by population
--> list of asian countries by gdp
--> list of argentine provinces by human development index
--> list of largest shopping centres in the nordic countries
--> list of tallest buildings in anchorage
--> list of most-visited art museums
--> list of tallest buildings in cape town
--> list of most-isolated mountains of switzerland
--> list of cities in hyōgo prefecture by population
--> list of stadiums in ireland by capacity
--> list of tallest dams
--> list of tallest buildings in st. petersburg, florida
--> list of counties of kenya by gross county product
--> list of hindi films of 1968
--> list of best-selling nintendo switch video games
--> list of tallest buildings in chihuahua city
--> list of tallest buildings in the waterloo regional municipality
--> list of top country albums number ones of 1993
--> districts of west bengal
--> list of the oldest newspapers
--> list of med

In [3]:
import random

In [42]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/candidate_tables.jsonl"))
cleaned_data = processor.process()

Filtering & Cleaning:   0%|          | 0/65565 [00:00<?, ?it/s]

Legend / Key Filter,496
Header Parsing Failures,"4,183"
Cell Content Too Long,"1,852"
Malformed Row Structures,"2,006"
Alphabetical Lists Removed,624
Long Tables Removed,"31,041"
Intra-Page Duplicates Removed,"10,531"


In [129]:
cnt = 0
ccc = 0
for s in cleaned_data:
    if "records" in s['table_name']['header_chain'].split(' > ')[0].lower():
        cnt += 1
        # print(f"--> {s['table_name']['header_chain']}")
        cols = list(s['rows'][0].keys())
        if any(["record" in a.lower() for a in cols]):
            ccc += 1
            # print(f"--> {s['table_name']['header_chain']}")
        else:
            print(f"--> {s['table_name']['header_chain']}")

print(cnt)
print(ccc)

--> List of Atletico Marte records and statistics > Achievements > Recopa de UNCAF/1991 Copa Centroamericano
--> List of Atletico Marte records and statistics > Players > Appearances
--> List of Atletico Marte records and statistics > Goalscorers > All-time top goalscorers (Clausura/Apertura) > Historical goals
--> List of Manly Warringah Sea Eagles records > Club Records > Win% vs Former Clubs
--> List of Manly Warringah Sea Eagles records > Club Records > Biggest Wins vs Former Clubs
--> List of Manly Warringah Sea Eagles records > Club Records > Biggest Losses vs Former Clubs
--> List of Manly Warringah Sea Eagles records > Club Records > Other Attendances
--> List of Melbourne Storm records > Individual Records > Points Scoring Records > Top 10 Most Points In a Game
--> List of Melbourne Storm records > Club Records > Losing Games > Top 10 Biggest Losses
--> List of Melbourne Storm records > Club Records > Winning Games > Top 10 Biggest Wins
--> List of Melbourne Storm records > Di

In [170]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(cleaned_data, 100))
await group_constructor.get_groups()

🚀 Processing 100 tables...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/100 [00:00<?, ?it/s]

Entity Column has Numeric/Date Values,11
Duplicate Entities Found,11
LLM Rejected,7


In [47]:
group_constructor.display_debug_samples(num_success=100)

In [156]:
from collections import Counter
ent_cols = [s['Entity column'].strip().lower() for s in group_constructor.valid_tables]
Counter(word for s in ent_cols for word in s.split())

Counter({'name': 92,
         'title': 45,
         'player': 19,
         'work': 14,
         'common': 10,
         'nominated': 9,
         'school': 9,
         'gold': 8,
         'team': 7,
         'film': 7,
         'club': 6,
         'award': 6,
         'nominee': 5,
         'institution': 5,
         'airline': 5,
         'station': 5,
         'airport': 4,
         'building': 4,
         'district': 4,
         'of': 4,
         'english': 4,
         'title)': 4,
         '/': 4,
         'ambassador': 4,
         'city': 4,
         'senator': 3,
         'company': 3,
         'ship': 3,
         'party': 3,
         'character': 3,
         'country': 3,
         'series': 3,
         'artist': 3,
         'conflict': 2,
         'wicket': 2,
         'national': 2,
         'region': 2,
         'in': 2,
         'malay': 2,
         'university': 2,
         'attorney': 2,
         'denomination': 2,
         'honor': 2,
         'athlete': 2,
         'chefs\\

In [152]:
aaaaaa = 0
bbbbbb = 0
cccccc = 0
for s in group_constructor.valid_tables:
    if 'name' in s['Entity column'].strip().lower():
        aaaaaa += 1
    elif s['Entity column'].strip().lower() in s['Group name'].strip().lower():
        bbbbbb += 1
        # print(f"{s['Group name']} --> {s['Entity column']}")
    else:
        cccccc += 1
        print(f"{s['Group name']} --> {s['Entity column']}")

print(aaaaaa)
print(bbbbbb)
print(cccccc)

NFL Network Exclusive Game Series results for the 2024 season --> Visiting team
Banks listed on the Athens Stock Exchange --> Company
Companies founded by Stanford University alumni before 1960 --> Company
Wars involving Mongolia and the Mengjiang --> Conflict
Awards and nominations received by Yuna at Anugerah Juara Lagu --> Nominated work
Governors of Kongo Central (1962–1972) --> Officeholder
Awards and nominations received by The Crown at the Hollywood Critics Association Television Awards --> Nominee
Top 10 highest-grossing Japanese films of 2006 --> Title
Online professional wrestling magazines and newsletters --> Title
Episodes of Natural World Series 35 --> Title
Awards and nominations received by Sonali Kulkarni at Zee Chitra Gaurav Puraskar --> Nominated work
Defunct colleges and universities in Delaware --> Institution
Members of the European Parliament for Slovakia (2004–2009) --> National party
Participants of the 1974 Karl Gotch Cup in New Japan Pro-Wrestling --> Masashi 

In [171]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidator

group_val = GroupValidator(random.sample(group_constructor.valid_tables, 50))
await group_val.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/50 [00:00<?, ?it/s]

In [172]:
from collections import Counter

rev = Counter([sample['verdict'] for sample in group_val.output_data])
print(f"{100 * rev['ACCEPT'] / sum(rev.values()):.1f}%")
rev

90.0%


Counter({'ACCEPT': 45,
         'FATEFULNESS': 2,
         'CORRECT_ENTITY': 1,
         'SPECIFICALITY': 1,
         'CLOSED_SET': 1})

In [173]:
for sample in group_val.output_data:
    members = [s[sample['Entity column']] for s in sample['Table data']]
    if sample['verdict'] != "ACCEPT":
        print(sample['Group name'])
        print(sample['Entity column'])
        print(f"{sample['verdict']} --> {sample['reason']}")
        print(sample['Wiki url'])
        print(members)
        print("-"*100)

Men's ice hockey players with multiple All-WCHA First Team appearances
Player
FATEFULNESS --> The entity list contains 'many player tied with', which is a table artifact and not a valid player name, making the group invalid.
https://en.wikipedia.org/wiki/List_of_men%27s_All-WCHA_Hockey_Teams#Multiple_appearances
['Marc Michaelis', 'Garry Bauman', 'Jason Blake', 'Jack Connolly', 'Tony Esposito', 'Greg Johnson', 'Keith Magnuson', 'John Mayasich', 'Bill Reichart', 'Brian Swanson', 'Ken Yackel', 'many player tied with']
----------------------------------------------------------------------------------------------------
21st-century vice presidents of the United States and their personal coats of arms
Name of vice president and blazon
CORRECT_ENTITY --> The Entity Column contains highly concatenated strings including the vice president's name, rank, term years, and the phrase 'No arms known', rather than just the primary entity names.
https://en.wikipedia.org/wiki/List_of_personal_coats_of_

In [164]:
accepted_samples = [sample for sample in group_val.output_data if sample['verdict']=='ACCEPT']

In [166]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidator

group_val = GroupValidator(accepted_samples)
await group_val.validate()

rev = Counter([sample['verdict'] for sample in group_val.output_data])
print(f"{100 * rev['ACCEPT'] / sum(rev.values()):.1f}%")
rev

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/89 [00:00<?, ?it/s]

92.1%


Counter({'ACCEPT': 82, 'CORRECT_ENTITY': 5, 'CLOSED_SET': 2})

In [169]:
for sample in group_val.output_data:
    members = [s[sample['Entity column']] for s in sample['Table data']]
    if sample['verdict'] == "CLOSED_SET":
        print(sample['Group name'])
        print(sample['Entity column'])
        print(f"{sample['verdict']} --> {sample['reason']}")
        print(sample['Wiki url'])
        print(members)
        print("-"*100)

Top 12 extinct Oceanian snails and slugs of the Holocene
Common name
CLOSED_SET --> The group name 'Top 12 extinct Oceanian snails and slugs' implies an arbitrary subset of a larger list, as there is no standard or objective ranking for extinct snails to form a 'Top 12'.
https://en.wikipedia.org/wiki/List_of_Oceanian_species_extinct_in_the_Holocene#FamilyPartulidae
['Toothed partula', "Captain Cook's bean snail", 'Rose-tipped partula snail', 'Miracle tree snail', 'Moorean smooth tree snail', 'Raiatean ground partula snail', 'Tahitian nodular partula', 'Pink partula', 'Sutural partula', 'Mount Tohiea tree snail', 'Mourning partula', 'Variable tree snail']
----------------------------------------------------------------------------------------------------
Fencing clubs in Australia
Fencing Club
CLOSED_SET --> The group name implies all fencing clubs in Australia, but the members only include university fencing clubs, making it an incomplete subset.
https://en.wikipedia.org/wiki/List_of_u

In [199]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidatorTest

group_val = GroupValidatorTest(random.sample(group_constructor.valid_tables, 4))
await group_val.validate()

rev = Counter([sample['verdict'] for sample in group_val.output_data])
print(f"{100 * rev[True] / sum(rev.values()):.1f}%")
rev

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/4 [00:00<?, ?it/s]

ChatCompletion(id='gen-1773190353-q2hzZ6AWwTuuDHm1Vo26', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='True', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, reasoning='**Evaluating Group Cohesion**\n\nI\'m currently focused on the coherence of the group "Wind power stations in Iran." Initial assessments suggest a common theme, but I need to ascertain if the group is specific enough to be meaningful. I\'m investigating if the shared location and function constitute a naturally occurring group, or if further refinement is necessary. The group size seems manageable for now.\n\n\n**Defining Group Membership**\n\nI\'ve confirmed each entity\'s presence within the specified geographic and functional criteria. The group\'s coherence is demonstrable, with all members adhering to the definition of "wind power stations in Iran." The entity column\'s members are valid, and the group definition provides a clea

Counter({None: 4})

In [184]:
for s in [sample for sample in group_val.output_data if not sample['verdict']]:
    print(s['Group name'])
    print(s['Entity column'])
    print(s['Wiki url'])
    print("-"*100)

Members of Task Force ✱/Suicide Squad Dark
Character
https://en.wikipedia.org/wiki/List_of_Suicide_Squad_members#Task_Force_%E2%9C%B1/Suicide_Squad_Dark
----------------------------------------------------------------------------------------------------
Top 5 mainland Chinese albums by annual sales percentage on the Billboard China V Chart in 2016
Album
https://en.wikipedia.org/wiki/List_of_Billboard_China_V_Chart_top_10_albums_of_2016#Mainland_%28Domestic%29_albums
----------------------------------------------------------------------------------------------------
Taiwanese films released in 2001
Title
https://en.wikipedia.org/wiki/List_of_Taiwanese_films_of_the_2000s#2001
----------------------------------------------------------------------------------------------------


In [187]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidator

group_val = GroupValidator([sample for sample in group_val.output_data if sample['verdict']])
await group_val.validate()

rev = Counter([sample['verdict'] for sample in group_val.output_data])
print(f"{100 * rev['ACCEPT'] / sum(rev.values()):.1f}%")
rev

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/17 [00:00<?, ?it/s]

Input tokens: 642
Output tokens: 188
Input tokens: 577
Output tokens: 271
Input tokens: 578
Output tokens: 207
Input tokens: 519
Output tokens: 255
Input tokens: 629
Output tokens: 236
Input tokens: 622
Output tokens: 251
Input tokens: 533
Output tokens: 369
Input tokens: 624
Output tokens: 406
Input tokens: 616
Output tokens: 430
Input tokens: 606
Output tokens: 548
Input tokens: 630
Output tokens: 182
Input tokens: 596
Output tokens: 217
Input tokens: 607
Output tokens: 155
Input tokens: 571
Output tokens: 238
Input tokens: 544
Output tokens: 219
Input tokens: 580
Output tokens: 892
Input tokens: 581
Output tokens: 181
94.1%


Counter({'ACCEPT': 16, 'CLOSED_SET': 1})

In [26]:
def format_sample_for_chat(s):
    cols = list(s['Table data'][0].keys())
    entity_col = s['Entity column']
    entities = [r.get(entity_col, "") for r in s['Table data']]

    return f"""
Group name: {s['Group name']}
Entity column: {entity_col}
Columns: {cols}
Entity members: {entities}
Group size: {len(entities)}
Source: {s['Wiki url']}
Row example: {s['Table data'][0]}
--------------------
""".strip()

In [48]:
for i in random.sample(group_constructor.valid_tables, 100):
    print(format_sample_for_chat(i))

Group name: International schools in Kowloon City District, Hong Kong
Entity column: Name (Abbreviation)
Columns: ['Name (Abbreviation)', 'Curriculum', 'Affiliation', 'Established']
Entity members: ['American International School Hong Kong (AIS)', 'Australian International School Hong Kong (AISHK)', 'Beacon Hill School (BHS)', 'Christian Alliance P.C. Lau Memorial International School (CAPCL)', 'Jockey Club Sarah Roe School (JCSRS)', 'King George V School, Hong Kong (KGV)', 'Kingston International School (Kingston)', 'Kowloon Junior School (KJS)', 'Po Leung Kuk Ngan Po Ling College (NPL)', 'Stamford American School Hong Kong (SAIS)', 'Think International School (TIS)', 'Yew Chung International School (YCIS)']
Group size: 12
Source: https://en.wikipedia.org/wiki/List_of_international_schools_in_Hong_Kong#Kowloon_City_District
Row example: {'Name (Abbreviation)': 'American International School Hong Kong (AIS)', 'Curriculum': 'AP', 'Affiliation': 'Americans in Hong Kong', 'Established': 1

In [ ]:
### ROLE
You are a strict Data Quality Auditor. Your task is to validate structured table groups extracted from Wikipedia.
You must determine whether each sample represents a clean, meaningful, and well-defined entity group suitable for a high-quality dataset.
Be conservative: reject doubtful samples rather than accepting them.

### INPUT DATA
- **Group Name:** The semantic description of the group extracted from the Wikipedia table or section.
- **Entity column:** The table column that represents the main entity type of the group.
- **Entity members:** The list of values from the entity column for all rows.
- **Group size:** Number of rows (entities) in the table.
- **Source:** The Wikipedia URL where the table was extracted.
- **Row example:** A full example row from the table.


### RULES FOR VALIDITY

1. **FATEFULNESS:**
   - The entity members must clearly match the concept described in the Group Name.
   - Reject if:
        - Entities do not logically belong to the group.
        - The group description is misleading or overly broad.
        - The entities represent multiple unrelated concepts.
    - Examples:
        - Group: European Rivers → Entities include cities. --> REJECT
        - Group: Famous Scientists → Entities include organizations. --> REJECT

2. **CLOSED_SET:**
   - Does this set represent a coherent and complete group?
   - Reject if:
        - The group appears to be a random subset of a larger list.
        - The group is ambiguous about why these specific rows were chosen.
        - *Exception:* If the Group Name specifies a filter (e.g., "Top 4 Scorers"), then a small set is valid. 
    - Examples:
        - Cities in France with only 6 random entries. --> REJECT
        - Top 10 Indian films of 2025 by box office collection. --> ACCEPT

3. **SPECIFICALITY:**
    - The group name must define a clear and narrow concept.
    - Reject if:
        - The group is too generic or vague.
        - The grouping rule is unclear.
    - Examples:
        - Famous movies --> REJECT
        - Important countries --> REJECT

4. **NATURALNESS:**
    - The group must represent a natural category people would reasonably group together.
    - Reject if:
        - The group looks artificially constructed.
        - The grouping rule feels arbitrary or meaningless.
    - Examples:
        - Cities whose names start with the letter Q and have more than 3 vowels --> REJECT
        - Countries with GDP between X and Y but excluding Z --> REJECT

5. **CORRECT_ENTITY:**
    - The Entity column must correctly represent the primary entity of the group.
    - Reject if:
        - The column contains metadata (rank, year, ID, etc.).
        - The column represents attributes rather than entities.
    - Examples:
        - Entity column: Rank --> REJECT
        - Entity column: Year --> REJECT

### OUTPUT FORMAT:
After analyzing all samples provide:
1. Dataset Quality Summary
    - Total samples: X
    - Valid samples: X
    - Invalid samples: X
    - Success rate: X%
2. If success rate < 95%:
    - List the main recurring problems, such as:
    - weak group definitions
    - incorrect entity columns
    - partial lists
    - unnatural groupings
    - Provide specific suggestions to improve dataset quality.

All samples are separated by a line, Evaluate each sample independently.

In [ ]:
f"""### ROLE
You are a strict Data Quality Auditor. Your task is to validate a structured table group extracted from Wikipedia. 
Determine if the sample represents a clean, meaningful, and well-defined entity group for a high-quality dataset.
Be conservative: reject doubtful samples rather than accepting them.

### SOURCE TRUST POLICY
Trust the provided Wikipedia Source and Group Name as the ground truth for the intent of the list. However, if the Entity Members or Column clearly contradict the source (e.g., the source is about "Rivers" but the entities are "Cities"), you must reject it.

### INPUT DATA
- **Group Name:** {GROUP_NAME}
- **Entity Column:** {ENTITY_COLUMN}
- **Entity Members:** {MEMBERS_LIST}
- **Group Size:** {SIZE}
- **Source URL:** {URL}
- **Row Example:** {EXAMPLE_ROW}

### RULES FOR VALIDITY

1. **FATEFULNESS:** Members must logically match the Group Name. Reject if the description is misleading, overly broad, or contains unrelated concepts.
2. **CLOSED_SET:** The group must be coherent and complete. Reject random subsets (e.g., "6 random cities in France"). 
   *Exception:* Specific filters (e.g., "Top 10", "Oldest 5", "Members in 2024") are valid.
3. **SPECIFICALITY:** The group must have a narrow, clear concept. Reject vague names like "Famous movies" or "Important countries."
4. **NATURALNESS:** The category must be one that humans would reasonably group together. Reject arbitrary rules (e.g., "Cities starting with Q that have 3 vowels").
5. **CORRECT_ENTITY:** The Entity Column must contain the primary entities, not metadata (Rank, Year, ID) or attributes.

### OUTPUT FORMAT
Provide only a single Python-style tuple containing the decision and the explanation. Do not include any other text, markdown formatting, or headers.

Format: ("DECISION", "EXPLANATION")
For Acceptances: ("ACCEPT", "Reason why the group is valid and coherent.")
For Rejections: ("RULE_NAME", "Specific explanation of the violation.")"""

# Test 3:

In [18]:
import re

def get_entity_col_deterministic(rows, page_title, section_header):
    """
    rows: The full list of dictionaries for the table.
    """
    if not rows or not isinstance(rows, list):
        return None
        
    # Correctly handle headers from the first row
    headers = list(rows[0].keys())
    page_ctx = (page_title + " " + section_header).lower()
    
    # 1. THE GRID FILTER (Pre-check)
    # Rejects tables that are just a matrix of years or stats
    grid_indicators = [r'^\d{4}$', r'^\d{4}–\d{4}$', r'^gold$', r'^silver$', r'^bronze$', r'^total$']
    num_grid_headers = sum(1 for h in headers if any(re.search(p, h.lower().strip()) for p in grid_indicators))
    
    if len(headers) > 0 and (num_grid_headers / len(headers)) > 0.6:
        return None

    # 2. THE KILL LIST
    kill_list = ['rank', 'gross', 'collection', 'revenue', 'year', 'date', 'pos', 'notes', 'ref', 'no.', 'total', 'index']
    
    # 3. PRIORITY WEIGHTS
    universal_priority = ['name', 'title', 'airline', 'team', 'character', 'honor', 'mayor', 'organization', 'mission']

    scores = {}
    for h in headers:
        h_low = h.lower().strip()
        score = 0
        
        # Rule A: Hard Block (Case-insensitive full word match)
        if any(k == h_low or f" {k}" in h_low or f"{k} " in h_low for k in kill_list):
            scores[h] = -100
            continue

        # Rule B: Lexical Anchor (Direct overlap with Page/Section context)
        # Matches "Airline" if "Airlines" is in the title
        ctx_words = re.findall(r'\w+', page_ctx)
        singular_h = h_low.rstrip('s')
        if singular_h in [w.rstrip('s') for w in ctx_words] and len(singular_h) > 2:
            score += 50

        # Rule C: Universal Match
        if h_low in universal_priority:
            score += 40

        # Rule D: Fuzzy Name Match (Catches 'Common name', 'Name in English')
        if 'name' in h_low:
            score += 30

        # Rule E: Data Content Validation (The "Peek" test)
        # We check the first 3 rows. If they are all numbers, it's not our entity.
        try:
            samples = [str(rows[i][h]) for i in range(min(len(rows), 3)) if rows[i].get(h)]
            if samples and all(s.replace('.', '', 1).isdigit() for s in samples):
                score -= 60 
        except:
            pass

        # Rule F: Position Bias (Wikipedia editors put the subject on the left)
        score += (len(headers) - headers.index(h)) * 1 

        scores[h] = score

    # Final Selection
    valid_scores = {k: v for k, v in scores.items() if v > 10}
    if not valid_scores:
        return None
        
    return max(valid_scores, key=valid_scores.get)

In [23]:
for s in [a for a in cleaned_data[:200] if a['table_name'].startswith("List of")]:
    # print(s['table_name'].split(' > ', 1)[0])
    print(s['table_name'].split('(', 1)[0])
    print(s['rows'][0].keys())
    page_title = s['table_name'].split(' > ', 1)[0]
    section_header = s['table_name'].split(' > ', 1)[-1].split('(', 1)[0]
    print(get_entity_col_deterministic(s['rows'], page_title, section_header))
    print(generate_deterministic_name(page_title, section_header))
    print("-" *100)

List of Social Distortion band members > Members > Current
dict_keys(['Name', 'Years active', 'Instruments', 'Release contributions'])
Name
> current Social Distortion band members
----------------------------------------------------------------------------------------------------
List of fictional birds > Charadriiformes
dict_keys(['Name', 'Species', 'Work', 'Notes'])
Name
Charadriiformes fictional birds
----------------------------------------------------------------------------------------------------
List of supermarket chains in Serbia > Speciality store chains > Cosmetics and healthcare
dict_keys(['Name', 'Stores', 'First store', 'Parent'])
Name
Speciality store > cosmetics and healthcare supermarket chains in Serbia
----------------------------------------------------------------------------------------------------
List of Supertramp band members > Members > Touring
dict_keys(['Name', 'Years active', 'Instruments', 'Release contributions'])
Name
> touring Supertramp band members

In [22]:
import re

def generate_deterministic_name(page_title, section_header):
    # 1. Clean the Page Title
    # Remove "List of", "List of the", etc.
    clean_title = re.sub(r'^list of (the )?', '', page_title, flags=re.IGNORECASE).strip()
    
    # 2. Clean the Section Header
    # Remove redundant info like "List of...", "(1990-2000)", or "Table"
    clean_section = re.sub(r'\(.*?\)', '', section_header).strip()
    clean_section = re.sub(r'^list of ', '', clean_section, flags=re.IGNORECASE).strip()
    
    # 3. Detect Redundancy
    # If the section header is "Current missions" and title is "Missions of Yemen"
    # we don't want "Current missions Missions of Yemen"
    words_in_title = set(clean_title.lower().split())
    words_in_section = clean_section.lower().split()
    
    # Filter out words from section that already exist in the title
    unique_section_words = [w for w in words_in_section if w not in words_in_title]
    unique_section = " ".join(unique_section_words).strip()

    # 4. Assembly
    if not unique_section or unique_section.lower() in ['list', 'table', 'current', 'general']:
        return clean_title.capitalize()
        
    return f"{unique_section.capitalize()} {clean_title}"

In [679]:
list_of_subset = [s for s in generator.valid_results if s['Wiki link'].split('/wiki/', 1)[-1].replace("_", " ").startswith("List of")]
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(list_of_subset, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/35 [00:00<?, ?it/s]

Degree
Doctor of Science and Technology
Doctor of Science
Doctor of Science
Doctor of Science
Doctor of Engineering
Honorary Doctorate
Doctor of Engineering
Doctor of Law
Doctor of Science


Villain,Outcome
Lord Randolph Hellebore,Bond and George Hellebore wreck his laboratory by smashing the serum tubes and igniting the paperwork.
Count Ugo Carnifex,His associate Zoltan floods his palace for his failure.
"Irina Sedova ""Babushka""",Bond destroys the machine.
Mrs. Glass,Succeeds.
El Huracán,Bond beats his obstacle course and escapes.
Dr. Perseus Friend,"Bond saves the king, destroys Friend's operation and forces Sedova to kill Friend."
"Irina Sedova ""Babushka""","Bond saves the king, destroys Friend's operation and forces Sedova to kill Friend."
Emil Lefebrve,Succeeds.
Caiboche,Succeeds.


Honor,Year
"Minister of Culture, Sports and Tourism Commendation",2016
‡ Presidential Commendation,2017
Hwagwan Order of Cultural Merit (5th class),2018
Special Presidential Envoy for Future Generations and Culture,2021
Game Changer Award,2020
James A. Van Fleet Award,2020
Presidential Commendation,2025


Ambassador,Assumed office
Nikki Haley,"January 27, 2017 (Confirmed 01/24/2017, 96–4)"
Kelly Craft,"September 12, 2019 (Confirmed 07/31/2019, 56–34)"
Jonathan R. Cohen,"June 8, 2018 (Confirmed 05/24/2018, voice vote)"
Richard M. Mills Jr.,"November 9, 2020 (Confirmed August 6, 2020, voice vote)"
Andrew Bremberg,"November 12, 2019 (Confirmed 10/22/2019, 50–44)"
Jackie Wolcott,"October 22, 2018 (Confirmed 09/24/2018, 75-19)"
Kip E. Tom,"May 22, 2019 (Confirmed 04/11/2019, voice vote)"
Cherith Norman Chalet,"October 17, 2018 (Confirmed 09/12/2018, voice vote)"
Kelley Eckels Currie,"August 2017 (Confirmed 08/03/2017, voice vote)"


In [681]:
tagger = ManualTagger(validator.approved_data, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
tagger.display() # Render the output area

📦 Progress Loaded: 10 samples already tagged in 'tagger_progress.json'.


Output()

In [543]:
from difflib import SequenceMatcher
import pandas as pd
from IPython.display import display, HTML

def audit_categorical_system(samples, fuzzy_threshold=0.9):
    categorical_strategies = ['UNIQUE_DIVERSITY', 'FREQUENCY_MODE', 'CATEGORICAL_FILTER_LOOKUP']
    
    # Filter targets
    target_samples = [s for s in samples if s.get('Selected strategy') in categorical_strategies]
    total_samples = len(target_samples)
    
    if total_samples == 0:
        print("🔍 No samples found for the specified categorical strategies.")
        return

    problematic_sample_indices = []
    column_stats = {}
    fail_details = []

    # --- PROCESSING ---
    for idx, sample in enumerate(target_samples):
        table = sample.get('Relevant table data', [])
        if not table or not isinstance(table, list): continue
        
        strat = sample.get('Selected strategy')
        all_cols = list(table[0].keys())
        if len(all_cols) <= 1: continue
        
        # Skip the first column (Entity Column)
        data_cols = all_cols[1:] 
        sample_has_issue = False
        
        for col in data_cols:
            if col not in column_stats:
                column_stats[col] = {'systemic_fails': 0, 'any_issue_count': 0}
            
            raw_values = [str(row.get(col, "")).strip() for row in table if row.get(col)]
            unique_vals = sorted(list(set(raw_values)), key=len)
            if len(unique_vals) < 2: continue
            
            issue_vals = set()
            for i in range(len(unique_vals)):
                for j in range(i + 1, len(unique_vals)):
                    v1, v2 = unique_vals[i].lower(), unique_vals[j].lower()
                    if v1 in v2 or SequenceMatcher(None, v1, v2).ratio() >= fuzzy_threshold:
                        issue_vals.update([unique_vals[i], unique_vals[j]])

            if issue_vals:
                column_stats[col]['any_issue_count'] += 1
                sample_has_issue = True
                
                # Check for Systemic Fail
                if len(issue_vals) == len(unique_vals):
                    column_stats[col]['systemic_fails'] += 1
                    fail_details.append({
                        'sample': sample,
                        'bad_col': col,
                        'bad_values': unique_vals
                    })

        if sample_has_issue:
            problematic_sample_indices.append(idx)

    # --- 1. THE BIG PICTURE SUMMARY ---
    clean_count = total_samples - len(problematic_sample_indices)
    
    display(HTML(f"""
    <div style="border: 2px solid #2c3e50; border-radius: 10px; padding: 20px; font-family: sans-serif; background-color: #f8f9fa;">
        <h2 style="margin-top: 0; color: #2c3e50;">📊 CATEGORICAL AUDIT: TOTAL PICTURE</h2>
        <div style="display: flex; gap: 40px;">
            <div style="text-align: center;">
                <h1 style="color: #27ae60; margin: 0;">{clean_count}</h1>
                <p style="color: #666;">CLEAN SAMPLES</p>
            </div>
            <div style="text-align: center;">
                <h1 style="color: #e74c3c; margin: 0;">{len(problematic_sample_indices)}</h1>
                <p style="color: #666;">PROBLEMATIC SAMPLES</p>
            </div>
            <div style="text-align: center; border-left: 1px solid #ccc; padding-left: 40px;">
                <h1 style="color: #2980b9; margin: 0;">{(clean_count/total_samples)*100:.1f}%</h1>
                <p style="color: #666;">SUCCESS RATE</p>
            </div>
        </div>
    </div>
    """))

    # --- 2. DRILL DOWN (SYSTEMIC FAILS) ---
    if fail_details:
        display(HTML("<h3 style='margin-top: 30px; color: #c0392b;'>🚨 SYSTEMIC FAILURES DETECTED</h3>"))
        
        for fail in fail_details:
            s = fail['sample']
            df_full = pd.DataFrame(s['Relevant table data'])
            
            # Highlight the problematic column in the display
            html_output = f"""
            <div style="margin-bottom: 40px; border-left: 5px solid #e74c3c; padding-left: 15px;">
                <p style="margin: 0;"><b>Strategy:</b> <span style="color: #2980b9;">{s['Selected strategy']}</span></p>
                <p style="margin: 5px 0;"><b>Question:</b> <span style="font-size: 1.1em; color: #2c3e50;">{s['Question']}</span></p>
                <p style="margin: 0; font-size: 0.9em; color: #c0392b;"><b>Issue:</b> Every value in column <b>'{fail['bad_col']}'</b> is a near-duplicate.</p>
                <div style="margin-top: 10px;">
                    {df_full.to_html(index=False)}
                </div>
            </div>
            <hr style="border: 0; border-top: 1px solid #eee;">
            """
            display(HTML(html_output))
    else:
        print("✅ No systemic failures found in the categorical attributes.")

# Usage
# audit_categorical_system(generator.valid_results)

In [596]:
audit_categorical_system(generator.valid_results)

Officeholder,Title
Marcel Tshishiku,Extraordinary commissioner
Georges Grenfell (b. 1908),President
Paul Isombuma,President
François Aradjabu,Extraordinary commissioner-general for the former province of Haut-Congo
Alphonse Abikwa,President
Ignace Michel Alamazani,"President (from 1965, governor)"
Pierre Efomi,Governor


Title,Films
Fantastic Four 2-Movie Collection,"Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"
The Ultimate Heroes Collection,"Daredevil , Elektra , Fantastic Four (2005), X-Men"
Marvel Heroes,"Daredevil , Elektra , Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer , X-Men , X2 , X-Men: The Last Stand"
Blu-ray 3-Pack,"Daredevil , Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"
20th Century Fox Triple Feature,"Daredevil , Fantastic Four (2005), X-Men"
20th Century Fox Triple Feature,"Daredevil , Elektra , Fantastic Four (2005)"
20th Century Fox Double Feature,"Fantastic Four (2005), X-Men"
Fantastic Four / Fantastic Four: Rise of the Silver Surfer,"Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"
Fantastic Four Double Feature,"Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"


In [570]:
cou = 0
tot = 0
sam = 0
for s in [a for a in generator.valid_results if a['Selected strategy'] == "TOP_K_RANKING"]:
    if "three" in s['Question'].lower():
       tot += 1
    sam += 1
print(f"Top 3: {tot/sam}")

Top 3: 1.0


In [557]:
cou = 0
tot = 0
sam = 0
for s in [a for a in generator.valid_results if a['Selected strategy'] == "AGGREGATION"]:
    if "total" in s['Question'].lower():
       tot += 1
    if "average" in s['Question'].lower():
       cou += 1
    sam += 1
print(f"Sum: {tot/sam}, Avg: {cou/sam}")

Sum: 0.9148936170212766, Avg: 0.0851063829787234


In [595]:
import pandas as pd
import re
from IPython.display import display, HTML

def audit_numeric_in_categorical(samples):
    categorical_strategies = ['UNIQUE_DIVERSITY', 'FREQUENCY_MODE', 'CATEGORICAL_FILTER_LOOKUP']
    target_samples = [s for s in samples if s.get('Selected strategy') in categorical_strategies]
    total_samples = len(target_samples)
    
    if total_samples == 0:
        print("🔍 No samples found for categorical strategies.")
        return

    problem_details = []
    problematic_sample_indices = set()

    # Regex to find: Integers, Decimals, and Ranges (e.g., 10-20, 10 to 20, 10/20)
    # This also handles currency symbols and commas
    num_pattern = re.compile(r'^[\$€£%]*\d+([.,]\d+)?([\s\-\/]+(to[\s\-]+)?\d+([.,]\d+)?[\$€£%]*)?$')

    for idx, sample in enumerate(target_samples):
        table = sample.get('Relevant table data', [])
        if not table or not isinstance(table, list): continue
        
        all_cols = list(table[0].keys())
        if len(all_cols) <= 1: continue
        
        data_cols = all_cols[1:]
        
        for col in data_cols:
            values = [str(row.get(col, "")).strip() for row in table if row.get(col)]
            if not values: continue
            
            # Find values that match numeric or range patterns
            numeric_values = [v for v in values if num_pattern.match(v.replace(',', ''))]

            # NEW THRESHOLD: 25% (1/4 of the column)
            if len(numeric_values) >= len(values) / 4:
                problematic_sample_indices.add(idx)
                problem_details.append({
                    'sample': sample,
                    'col': col,
                    'found_nums': list(set(numeric_values))
                })
                break 

    # --- 1. THE BIG PICTURE SUMMARY ---
    clean_count = total_samples - len(problematic_sample_indices)
    
    display(HTML(f"""
    <div style="border: 2px solid #8e44ad; border-radius: 10px; padding: 20px; font-family: sans-serif; background-color: #f4f0f7;">
        <h2 style="margin-top: 0; color: #8e44ad;">🔢 NUMERIC & RANGE AUDIT (Threshold: 25%)</h2>
        <div style="display: flex; gap: 40px;">
            <div style="text-align: center;">
                <h1 style="color: #27ae60; margin: 0;">{clean_count}</h1>
                <p style="color: #666;">CATEGORICAL PURITY</p>
            </div>
            <div style="text-align: center;">
                <h1 style="color: #e74c3c; margin: 0;">{len(problematic_sample_indices)}</h1>
                <p style="color: #666;">NUMERIC/RANGE LEAKS</p>
            </div>
            <div style="text-align: center; border-left: 1px solid #ccc; padding-left: 40px;">
                <h1 style="color: #8e44ad; margin: 0;">{(clean_count/total_samples)*100:.1f}%</h1>
                <p style="color: #666;">OVERALL QUALITY</p>
            </div>
        </div>
    </div>
    """))

    # --- 2. EXAMPLES (QUESTION + FULL TABLE) ---
    if problem_details:
        display(HTML("<h3 style='margin-top: 30px; color: #e67e22;'>⚠️ LEAKAGE EXAMPLES (Including Ranges)</h3>"))
        for p in problem_details:
            s = p['sample']
            df_full = pd.DataFrame(s['Relevant table data'])
            
            html_output = f"""
            <div style="margin-bottom: 40px; border-left: 5px solid #e67e22; padding-left: 15px;">
                <p style="margin: 0;"><b>Strategy:</b> {s['Selected strategy']}</p>
                <p style="margin: 5px 0;"><b>Question:</b> <span style="font-size: 1.1em; color: #2c3e50; font-weight: bold;">{s['Question']}</span></p>
                <p style="margin: 0; color: #d35400;"><b>Issue:</b> Column <b>'{p['col']}'</b> leaked numeric values or ranges: <code style="background: #fff; padding: 2px 5px;">{p['found_nums']}</code></p>
                <div style="margin-top: 10px;">
                    {df_full.to_html(index=False)}
                </div>
            </div>
            <hr style="border: 0; border-top: 1px solid #eee;">
            """
            display(HTML(html_output))
    else:
        print("✅ No numeric or range leakage detected in categorical columns!")

In [594]:
audit_numeric_in_categorical(generator.valid_results)

College Name,NIRF Rank (2025)
Kristu Jayanti (Deemed to be University ),34
"M S Ramaiah College of Arts, Science, and Commerce",67
St Aloysius (Deemed to be University),73
St. Joseph's College of Commerce,98
Mount Carmel College,NIL
Jyoti Nivas College,NIL
NMKRV College for Women,NIL
"MES College of Arts, Commerce and Science",NIL
Maharani Lakshmi Ammanni College for Women,NIL
Seshadripuram College,NIL


# Test for entities:

In [143]:
from collections import Counter
num_tot = len(cleaned_data)
num_lo = []
ent_h = []
temp_num = 0
for s in cleaned_data:
    wiki_title = s['table_name'].split(' > ')[0].split('List of ')[1].lower()
    for h in s['rows'][0].keys():
        if h.lower() in [a.lower() for a in TEST_KEYWORDS] or h.lower() in universal_priority or 'name' in h.lower():
            temp_num += 1
    if temp_num == 5:
        pass
        print(f"{wiki_title} --> {s['rows'][0].keys()}")
    num_lo.append(temp_num)
    if temp_num == 1:
        for h in s['rows'][0].keys():
            if h.lower() in [a.lower() for a in TEST_KEYWORDS]:
                # print(f"{wiki_title} --> {h}")
                ent_h.append(h)
    temp_num = 0
print(Counter(num_lo))
print(len(set(ent_h)))
# set(ent_h)

ncaa division i institutions --> dict_keys(['School', 'Common name', 'Nickname', 'City', 'State', 'Type', 'Subdivision', 'Former Div.', 'Conference', 'Reclassified', 'Full member'])
bridges in croatia --> dict_keys(['English name', 'Croatian name', 'Distinction', 'Length', 'Type', 'Carries Crosses', 'Opened', 'Location', 'County'])
cities, towns and villages in cyprus --> dict_keys(['Name ( en )', 'District', 'Population ( urban )(2021)', 'Area ( urban )', 'Name ( gr )', 'Name ( tr )', 'Other name(s)'])
Counter({1: 2878, 0: 1432, 2: 1108, 3: 223, 4: 46, 5: 3})
75


In [148]:
from knowledge_maze.create_dataset.create_questions.visualization import WikiTableCleaner

# ==========================================
# USAGE EXAMPLE
# ==========================================

# Run the processor
cleaner = WikiTableCleaner()
new_dataset = cleaner.process_batch(cleaned_data)

# # Print results
# import json

# print(json.dumps(new_dataset, indent=2, ensure_ascii=False))


                         WIKI TABLE PROCESSING SUMMARY                          
Total Tables Processed:   5690
Passed (Closed Sets):     5099
Rejected:                 591
Success Rate:             89.6%
--------------------------------------------------------------------------------
                                EXTRACTED GROUPS                                
--------------------------------------------------------------------------------
 1. Current Social Distortion band members                                           | Column: Name                 | Rows: 4
 2. Speciality store Cosmetics and healthcare supermarket chains in Serbia           | Column: Name                 | Rows: 11
 3. Touring Supertramp band members                                                  | Column: Name                 | Rows: 5
 4. Box office collection Tamil films of 2019                                        | Column: Title                | Rows: 10
 5. Former programming Blocks programs broadc

In [149]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(new_dataset)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/5099 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/5099 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     5025
❌ Groups Discarded:  74
📈 Success Rate:      98.5%
-------------------------------------------------------
Strategy Distribution (Top 4 per table):
  - UNIQUE_DIVERSITY          : 3722
  - COUNT_TOTAL               : 2493
  - EXTREME_TEMPORAL          : 2362
  - FILTER_COUNT_TEMPORAL     : 1913
  - FREQUENCY_MODE            : 1328
  - CATEGORICAL_FILTER_LOOKUP : 988
  - EXTREME_NUMERICAL         : 553
  - TOP_K_RANKING             : 471
  - AGGREGATION               : 427
  - COMPARATIVE_DIFFERENCE    : 363
  - FILTER_COUNT_NUMERICAL    : 266
-------------------------------------------------------
POSSIBLE STRATEGIES  | GROUP COUNT  | PERCENTAGE
-------------------------------------------------------
#1                   | 639          |      12.7%
#2                   | 1168         |    

In [150]:
import random
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

generator = QuestionGenerator(random.sample(selector.updated_data, 100),
                              strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await generator.generate_questions()

🚀 Starting Question Generation for 100 tables...
--- Round 1: Processing 100 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/100 [00:00<?, ?it/s]

--- Round 2: Processing 2 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/2 [00:00<?, ?it/s]

--- Round 3: Processing 1 tables ---


...

-----------------------------------------------------------------

[2] Table: Key Boston Bruins players
    Original Data: https://en.wikipedia.org/wiki/List_of_Boston_Bruins_players
    Strategy Attempted: COUNT_TOTAL
    Rejection Reason:   The data provided does not include any distinct entities (players) to count, only statistical categories are listed.
    Table Data:


,W
0,SO
1,L
2,GAA
3,T
4,OTL a
5,SV%


-----------------------------------------------------------------

[3] Table: Collection films at the 2007 Sundance Film Festival
    Original Data: https://en.wikipedia.org/wiki/List_of_films_at_the_2007_Sundance_Film_Festival
    Strategy Attempted: FILTER_COUNT_TEMPORAL
    Rejection Reason:   The 'Year' column contains non-temporal data and does not allow for a valid count of films within a specific year range.
    Table Data:


,Title,Year
0,River's Edge,1986
1,River's Edge,Written by Neal Jimenez
2,River's Edge,"Starring Crispin Glover , Keanu Reeves , Ione ..."
3,X: The Unheard Music,1986
4,X: The Unheard Music,Written by W. T. Morgan
5,X: The Unheard Music,"Starring Exene Cervenka , John Doe , Billy Zoo..."


-----------------------------------------------------------------


In [151]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(generator.valid_results, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/85 [00:00<?, ?it/s]

Crater
Catalina
Fuchinobe
Gando
Hammaguira
Kamisunagawa
Kamoi
Komaba
Laurel
Miyabaru
San Marco


Name,Years
Antonio Napolitano,None
Luca Biolato,None
Enrico De Maio,None
Giovanni Ferrero,None
Pietro Cordone,None
Ranieri Fornari,None
Domenico Pedata,None
Paolo Dionisi,None
Giorgio Starace,None
Liborio Stellino,None


Entity/Organization
European Union
Hong Kong
Taiwan
United Nations


Power station,Max. capacity MW
Cunderdin,128.00
Ambri Solar Farm,1.00
Badgingarra Solar Farm,19.25
Emu Downs Solar Farm,20.00
Esperance Solar Farm,4.00
Garden Island,2.00
Greenough River Solar Farm,40.00
Merredin Solar Farm,100.00
Northam Solar Farm,10.00


Award,Year
Best Actress,2006
Best Actress,2009
Best Actress,2010
Best Actress,2011
Best Actress,2012
Best Actress,2013
Best Actress,2015
Best Actress,2016
Best Actress,2017
Best Actress,2018


Team,Pts
Free agent,992
Dallas Stars,956
Los Angeles Kings,946
Carolina Hurricanes,910


In [152]:
from knowledge_maze.create_dataset.create_questions.question_validator import ManualTagger

tagger = ManualTagger(validator.approved_data, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
tagger.display() # Render the output area

📦 Progress Loaded: 0 samples already tagged in 'tagger_progress.json'.


Output()

In [54]:
validated_tables[0]

{'table_name': 'White City: A Novel > Track listing (Side one)',
 'rows': [{'No.': 1.0, 'Title': '" Give Blood "', 'Length': '5:44'},
  {'No.': 2.0, 'Title': '"Brilliant Blues"', 'Length': '3:06'},
  {'No.': 3.0, 'Title': '" Face the Face "', 'Length': '5:51'},
  {'No.': 4.0, 'Title': '"Hiding Out"', 'Length': '3:00'},
  {'No.': 5.0, 'Title': '"Secondhand Love"', 'Length': '4:12'}],
 'density': 1.0}

In [62]:
validated_tables = load_jsonl(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
list_of_validated_tables = [a for a in validated_tables if a['table_name'].startswith("List of")]
save_jsonl("final_tables_list_of", list_of_validated_tables)
processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables_list_of.jsonl"))
cleaned_data = processor.process()

Filtering & Cleaning:   0%|          | 0/41609 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      41609
Dropped (Bad Headers):     1659
Dropped (Cell too long):   1628
Dropped (Repeat Rows):     802
Short tables found:        17514
Unique source pages:       5690
Final sampled (1/page):    5690
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------

═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

,Name,National party,EP group
0,Georges Bach,Christian Social People's Party (CSV),EPP
1,Mady Delvaux-Stehres,Luxembourg Socialist Workers' Party (LSAP),S&D
2,Frank Engel,Christian Social People's Party (CSV),EPP
3,Charles Goerens,Democratic Party (DP),ALDE
4,Christophe Hansen (2018–2019),Christian Social People's Party (CSV),EPP
5,Tilly Metz (2018–2019),The Greens,Greens/EFA
6,Viviane Reding (2014–2018),Christian Social People's Party (CSV),EPP
7,Claude Turmes (2014–2018),The Greens,Greens/EFA



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
List of governors of Danish India > List > Governors of Dannemarksnagore (List of governors of Dannemarksnagore)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Tenure,Incumbent,Notes
0,1698–1699,Andreas Andræ,Concluded the Dano-Mughal Treaty together with...
1,1699–1702,Thomas Schmertz,Concluded the Dano-Mughal Treaty together with...
2,1702–1706,Johan Michelsen,
3,1706–1709,Jacob Panch,
4,1709–1711,Wolf Ravn,
5,1711–1714,Rasmus Attrup,Left the fort because of Mughal holstilities
6,1718,Gallus Bøgvad,



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
List of WNBL awards > Individual awards
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Award,Created,Description,Most recent winner(s)
0,Most Valuable Player ( Suzy Batkovic Medal ),1982,Awarded to the best performing player of the W...,Stephanie Talbot (2020)
1,Grand Final Most Valuable Player ( Rachael Spo...,1985,Awarded to the best performing player of the W...,Leilani Mitchell (2020)
2,Defensive Player of the Year ( Robyn Maher Med...,1990,Awarded to the best defensive player of the WN...,Stephanie Talbot (2020)
3,Youth Player of the Year (Betty Watson Medal),1988,Awarded to the best performing player 23 years...,Shyla Heal (2020)
4,Sixth Woman of the Year,2020,Awarded to the most incredible talent coming o...,Zitina Aokuso (2020)
5,Coach of the Year,1987,Awarded to the best coach of the WNBL season.,Shannon Seebohm (2020)
6,Leading Scorer Award,1981,Awarded to the player with the highest average...,Liz Cambage (2020)
7,Leading Rebounder Award,2020,Awarded to the player with the highest average...,Anneli Maley (2020)
8,Golden Hands Award,2020,Awarded to the player with the highest Golden ...,Leilani Mitchell (2020)


In [67]:
import re

class ClosedSetExtractor:
    def __init__(self):
        # Keywords that signal an "Open" or "Subjective" set (100% Precision Kill-switch)
        self.open_set_triggers = [
            r'\bnotable\b', r'\bfamous\b', r'\bselected\b', 
            r'\bmajor\b', r'\bexamples?\b', r'\bpartial\b'
        ]
        # High-confidence entity identifiers
        self.universal_priority = [
            'name', 'title', 'airline', 'team', 'character', 'honor', 
            'mayor', 'organization', 'mission', 'planet', 'film', 'company'
        ]
        # Statistical columns to reject
        self.kill_list = [
            'rank', 'gross', 'collection', 'revenue', 'year', 'date', 
            'pos', 'notes', 'ref', 'no.', 'total', 'index', 'rank'
        ]
        
        self.stats = {"total_processed": 0, "passed": 0, "rejected_open_set": 0, "rejected_no_entity": 0, "incomplete_set": 0}

    def _is_closed_set(self, page_title, hierarchy):
        """Determines if the context implies a complete, exhaustive list."""
        context = (page_title + " " + " ".join(hierarchy)).lower()
        for pattern in self.open_set_triggers:
            if re.search(pattern, context):
                return False
        return True

    def _extract_entity_col(self, rows, page_title, hierarchy):
        """Deterministic Scorer from our previous iteration."""
        if not rows: return None
        headers = list(rows[0].keys())
        page_ctx = (page_title + " " + " ".join(hierarchy)).lower()
        
        scores = {}
        for h in headers:
            h_low = h.lower().strip()
            # Rule 1: Kill list
            if any(k == h_low or f" {k}" in h_low or f"{k} " in h_low for k in self.kill_list):
                continue
            
            score = 0
            # Rule 2: Lexical Anchor (Title match)
            singular_h = h_low.rstrip('s')
            if singular_h in page_ctx and len(singular_h) > 2:
                score += 50
            # Rule 3: Universal Labels
            if h_low in self.universal_priority:
                score += 40
            # Rule 4: Data Check (Numbers check)
            try:
                samples = [str(rows[i][h]) for i in range(min(len(rows), 3)) if rows[i].get(h)]
                if samples and all(s.replace('.', '', 1).isdigit() for s in samples):
                    score -= 60
            except: pass
            
            # Rule 5: Position
            score += (len(headers) - headers.index(h))
            scores[h] = score

        return max(scores, key=scores.get) if scores and max(scores.values()) > 0 else None

    def _generate_group_name(self, page_title, hierarchy):
        """Fuses Hierarchy and Title while removing redundancy."""
        clean_title = re.sub(r'^list of (the )?', '', page_title, flags=re.IGNORECASE).strip()
        # Filter hierarchy to remove redundancy with the title
        title_words = set(clean_title.lower().split())
        filtered_hierarchy = []
        for h in hierarchy:
            h_clean = re.sub(r'\(.*?\)', '', h).strip()
            h_words = h_clean.split()
            unique_words = [w for w in h_words if w.lower() not in title_words]
            if unique_words:
                filtered_hierarchy.append(" ".join(unique_words))
        
        if not filtered_hierarchy:
            return clean_title.capitalize()
        return f"{' '.join(filtered_hierarchy).capitalize()} {clean_title}"

    def process(self, page_title, hierarchy, table_dict):
        """
        Input: 
            page_title (str)
            hierarchy (list of strings: [h2, h3...])
            table_dict (list of dicts)
        """
        self.stats["total_processed"] += 1
        
        # 1. Check if the set is defined as "Closed"
        if not self._is_closed_set(page_title, hierarchy):
            self.stats["rejected_open_set"] += 1
            return None

        # 2. Identify Entity Column
        entity_col = self._extract_entity_col(table_dict, page_title, hierarchy)
        if not entity_col:
            self.stats["rejected_no_entity"] += 1
            return None

        # 3. Generate the Set Name
        group_name = self._generate_group_name(page_title, hierarchy)

        for row in table_dict:
            val = str(row.get(entity_col, "")).strip().lower()
    
            if val and val not in ["", "none", "null", "n/a", "na", "-", "—", "unknown", "tba", "tbd"]:
                # The member is valid, proceed with extraction
                # print(val + " - OK")
                pass
            else:
                # Skip this row or mark the table as potentially incomplete
                # print(val + " - PROBLEM")
                self.stats["incomplete_set"] += 1
                return None
        
        self.stats["passed"] += 1
        return {
            "group_name": group_name,
            "entity_column": entity_col,
            "members": [t[entity_col] for t in table_dict],
            "status": "CLOSED_SET_VERIFIED",
            "table": table_dict
        }

    def get_summary(self):
        return (f"--- Processing Summary ---\n"
                f"Total Tables: {self.stats['total_processed']}\n"
                f"Passed (Closed Sets): {self.stats['passed']}\n"
                f"Rejected (Subjective/Open): {self.stats['rejected_open_set']}\n"
                f"Rejected (No clear Entity): {self.stats['rejected_no_entity']}\n"
                f"Rejected (Incomplete Set): {self.stats['incomplete_set']}\n"
                f"Precision Estimate: 100% (Deterministic)")

In [70]:
import pandas as pd
extractor = ClosedSetExtractor()

# Assuming 'all_tables' is your list of table objects
for s in cleaned_data[:100]:
    # 1. Prepare the inputs from your data structure
    current_rows = s['rows'] 
    
    # Splitting the title and hierarchy from your 'table_name' string
    # e.g., "List of defunct airlines of Oceania > Norfolk Island"
    parts = s['table_name'].split(' > ')
    page_title = parts[0]
    hierarchy = parts[1:] if len(parts) > 1 else []

    # 2. Run the extractor
    result = extractor.process(
        page_title=page_title,
        hierarchy=hierarchy,
        table_dict=current_rows
    )

    # 3. Handle the result
    if result:
        print(f"Set Verified: {result['group_name']}")
        print(f"Entity Col:   {result['entity_column']}")
        print(f"Entities:    {result['members']}")
        print("Table Data:")
        display(pd.DataFrame(result['table']))
        
        print("-" * 30)

# See the final tally
print(extractor.get_summary())

Set Verified: Current Social Distortion band members
Entity Col:   Name
Entities:    ['Mike Ness', 'Jonny "2 Bags" Wickersham', 'Brent Harding', 'David Hidalgo Jr.']
Table Data:


,Name,Years active,Instruments,Release contributions
0,Mike Ness,1978–present,lead vocals lead guitar,all Social Distortion releases
1,"Jonny ""2 Bags"" Wickersham",2000–present,rhythm guitar backing vocals,"all Social Distortion releases from Sex, Love ..."
2,Brent Harding,2005–present,bass backing vocals,"all Social Distortion releases from "" Far Behi..."
3,David Hidalgo Jr.,2010–present,drums percussion,""" Up Around the Bend "" (2013) Machine Gun Melo..."


------------------------------
Set Verified: Speciality store cosmetics and healthcare supermarket chains in Serbia
Entity Col:   Name
Entities:    ['BENU', 'Lilly', 'DM', 'Jasmin', 'The Body Shop', 'KIKO', 'Sephora', "L'Occitane en Provence", "Kiehl's", 'Yves Rocher', 'Lush']
Table Data:


,Name,Stores,First store,Parent
0,BENU,451,2010,Phoenix Pharmahandel
1,Lilly,234,2006,Lilly drogerie
2,DM,132,2004,DM
3,Jasmin,27,1956,Jasmin parfimerije
4,The Body Shop,11,2019,Natura & Co
5,KIKO,9,2021,KIKO Milano
6,Sephora,5,2010,LVMH
7,L'Occitane en Provence,5,2012,L'occitane
8,Kiehl's,4,2019,L'Oréal
9,Yves Rocher,1,2008,Yves Rocher


------------------------------
Set Verified: Touring Supertramp band members
Entity Col:   Name
Entities:    ['Scott Page', 'Fred Mandel', 'Marty Walsh', 'Brad Cole', 'Steve Reid']
Table Data:


...

------------------------------
Set Verified: By country arabian gulf countries highest-grossing Indian films in overseas markets
Entity Col:   Title
Entities:    ['Jawan', 'Pathaan', 'Baahubali 2: The Conclusion', 'L2: Empuraan', 'Bajrangi Bhaijaan', 'Dangal', 'Sultan', 'Lokah Chapter 1: Chandra', 'Dilwale', 'KGF: Chapter 2']
Table Data:


,Rank,Title,Gross ( USD ),Language(s),Year
0,1,Jawan,"$17,700,000",Hindi,2023
1,2,Pathaan,"$14,200,000",Hindi,2023
2,3,Baahubali 2: The Conclusion,"$10,310,000",Telugu,2017
3,4,L2: Empuraan,"$9,719,000",Malayalam,2025
4,5,Bajrangi Bhaijaan,"$9,450,000",Hindi,2015
5,6,Dangal,"$8,800,000",Hindi,2016
6,7,Sultan,"$8,570,000",Hindi,2016
7,8,Lokah Chapter 1: Chandra,"$8,510,000",Malayalam,2025
8,9,Dilwale,"$8,410,000",Hindi,2015
9,10,KGF: Chapter 2,"$8,150,000",Kannada,2022


------------------------------
Set Verified: Boston music awards and nominations received by Ray LaMontagne
Entity Col:   Award
Entities:    ['Act of the Year', 'Male Vocalist of the Year', 'Best Male Singer-Songwriter', 'Album of the Year', 'Song of the Year', 'Act of the Year', 'Male Vocalist of the Year', 'Album of the Year']
Table Data:


,Year,Nominee / work,Award,Result
0,2005,Ray LaMontagne,Act of the Year,Nominated
1,2005,Ray LaMontagne,Male Vocalist of the Year,Nominated
2,2005,Ray LaMontagne,Best Male Singer-Songwriter,Won
3,2005,Trouble,Album of the Year,Won
4,2005,""" Trouble """,Song of the Year,Won
5,2008,Ray LaMontagne,Act of the Year,Nominated
6,2008,Ray LaMontagne,Male Vocalist of the Year,Nominated
7,2008,Gossip in the Grain,Album of the Year,Nominated


------------------------------
--- Processing Summary ---
Total Tables: 100
Passed (Closed Sets): 92
Rejected (Subjective/Open): 5
Rejected (No clear Entity): 2
Rejected (Incomplete Set): 1
Precision Estimate: 100% (Deterministic)


# Test for caption finding:

In [181]:
for s in cleaned_data:
    if "(" in s['table_name'].lower():
        print(f"{s['table_name']} --> {s['table_name'].split('(')[-1].split(')')[0]}")

List of fictional birds > Charadriiformes(gulls, terns, auks, and waders) --> gulls, terns, auks, and waders
List of Tamil films of 2019 > Box office collection (Highest worldwide gross of 2019) --> Highest worldwide gross of 2019
List of 2012 box office number-one films in Japan > Highest-grossing films (Highest-grossing films of 2012) --> Highest-grossing films of 2012
List of Cardiff City F.C. seasons > Key (Key to colours and symbols:) --> Key to colours and symbols:
List of Chinese films of 2014 > Box office (Highest-grossing domestic films of 2014 in China) --> Highest-grossing domestic films of 2014 in China
List of athletes from Montana > Basketball (Basketball players from Montana) --> Basketball players from Montana
List of documentary films > Z (Contents : Top 0–9 A B C D E F G H I J K L M N O P Q R S T U V W X Y Z See also References) --> Contents : Top 0–9 A B C D E F G H I J K L M N O P Q R S T U V W X Y Z See also References
List of 2010 box office number-one films in Sp

In [196]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

# extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/final_tables_test.jsonl"))
# validated_tables_file_name = await extractor.run_with_progress(random.sample(table_titles, 1000))

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables_test.jsonl"))
ttttt_cleaned_data = processor.process()

Filtering & Cleaning:   0%|          | 0/1770 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      1770
Dropped (Bad Headers):     152
Dropped (Cell too long):   48
Dropped (Repeat Rows):     38
Short tables found:        810
Unique source pages:       312
Final sampled (1/page):    312
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
{'h

,Programs,Category,Network,Number of Awards
0,The Bold Ones: The Senator,Drama,NBC,4
1,The Mary Tyler Moore Show,Comedy,CBS,4
2,All in the Family,Comedy,CBS,3
3,The Price,Special,NBC,3
4,The Andersonville Trial,Special,PBS,2
5,The Flip Wilson Show,Variety,NBC,2
6,Singer Presents Burt Bacharach,Variety,CBS,2
7,Wide World of Sports,Sports,ABC,2



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
{'header_chain': '2019 Melon Music Awards > Judging criteria', 'caption': None, 'additional_text': None}
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Division,Online Voting,Digital Sales,Judge Score
0,Top 10 Artists,20%,80%,—
1,Main awards*,20%,60%,20%
2,Genre Awards**,30%,40%,30%
3,Popularity Awards***,60%,40%,—
4,Special Awards****,—,—,100%



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
{'header_chain': 'Tsinghua University > Academics > Rankings and reputation > Subjects rankings', 'caption': 'THE World University Rankings by Subject 2024', 'additional_text': None}
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Subject,Global,National
0,Arts & humanities,34,2
1,Business & economics,8,1
2,Clinical & health,14,1
3,Computer science,12,1
4,Education,7,1
5,Engineering,15,2
6,Law,33,1
7,Life sciences,=12,1–2
8,Physical sciences,14,1
9,Social sciences,21,2


In [199]:
for s in ttttt_cleaned_data:
    if s['table_name']['header_chain'].lower().startswith("list of"):
        if s['table_name']['caption'] or s['table_name']['additional_text']:
            print(s['table_name'])
            # print("-"*100)

{'header_chain': 'List of mayors of Naples > Deputy Mayor', 'caption': None, 'additional_text': 'The Deputy Mayor is nominated and eventually dismissed by the Mayor.'}
{'header_chain': 'List of power stations in Angola > Hydroelectric', 'caption': None, 'additional_text': 'List of all hydroelectric power plants in Angola.'}
{'header_chain': 'List of Colorado Avalanche players > Key', 'caption': 'Abbreviations', 'additional_text': None}
{'header_chain': 'List of mayors of Clermont-Ferrand > Elected mayors', 'caption': None, 'additional_text': 'The mayors of Clermont-Ferrand from 1893 to present.'}
{'header_chain': 'List of Chicago Blackhawks players > Key', 'caption': 'Skaters', 'additional_text': None}
{'header_chain': "List of awards and nominations received by Jonny Greenwood > Critics' Choice Movie Awards", 'caption': None, 'additional_text': "The Critics' Choice Movie Awards are presented annually since 1995 by the Broadcast Film Critics Association for outstanding achievements in 

In [226]:
from knowledge_maze.create_dataset.create_questions.visualization import WikiTableCleaner

# ==========================================
# USAGE EXAMPLE
# ==========================================

# Run the processor
cleaner = WikiTableCleaner()
new_dataset = cleaner.process_batch([a for a in ttttt_cleaned_data if a['table_name']['header_chain'].lower().startswith("list of")])

# # Print results
# import json

# print(json.dumps(new_dataset, indent=2, ensure_ascii=False))


                              WIKI TABLE PROCESSING SUMMARY                               
Total Tables Processed:   95
Passed (Closed Sets):     87
Rejected:                 8
Success Rate:             91.6%
------------------------------------------------------------------------------------------
                                     EXTRACTED GROUPS                                     
------------------------------------------------------------------------------------------
 1. List of schools in the Northern Territory | Cols: ['Name', 'Suburb', 'LGA'] | Rows: 7
 2. List of mayors of Naples                 | Cols: ['Mayor', 'Deputy', 'Term start', 'Term end', 'Party'] | Rows: 12
 3. List of airlines of Cambodia             | Cols: ['Airline', 'IATA', 'ICAO', 'Callsign'] | Rows: 4
 4. List of ambassadors of Russia to France  | Cols: ['Name', 'Title']         | Rows: 9
 5. List of airlines of the Philippines      | Cols: ['Airline', 'Hubs and focus cities'] | Rows: 12
 6. List of pow

In [210]:
new_dataset[0].keys()

dict_keys(['Table context', 'Wiki link', 'Table data'])

In [227]:
for s in new_dataset:
    if s['Table context']['header_chain'].lower().startswith("list of"):
        if s['Table context']['caption'] or s['Table context']['additional_text']:
            for k,v in s['Table context'].items():
                print(f"{k}: {v}")
            print("-"*100)

header_chain: List of mayors of Naples > Deputy Mayor
caption: None
additional_text: The Deputy Mayor is nominated and eventually dismissed by the Mayor.
possible_entity_cols: ['Mayor', 'Deputy', 'Term start', 'Term end', 'Party']
----------------------------------------------------------------------------------------------------
header_chain: List of power stations in Angola > Hydroelectric
caption: None
additional_text: List of all hydroelectric power plants in Angola.
possible_entity_cols: ['Location', 'Station', 'River']
----------------------------------------------------------------------------------------------------
header_chain: List of Colorado Avalanche players > Key
caption: Abbreviations
additional_text: None
possible_entity_cols: ['Nat', 'Nationality']
----------------------------------------------------------------------------------------------------
header_chain: List of mayors of Clermont-Ferrand > Elected mayors
caption: None
additional_text: The mayors of Clermont-Fe

In [243]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(new_dataset)
await group_constructor.get_groups()

🚀 Grouping 87 tables using Pre-Filtered Context...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/87 [00:00<?, ?it/s]


══════════════════════════════════════════════════════════════════════════════════════════
                                    📊 GROUPING REPORT                                     
══════════════════════════════════════════════════════════════════════════════════════════
  ✅ Valid Groups Extracted: 77
  ❌ Invalid/Rejected:       10
══════════════════════════════════════════════════════════════════════════════════════════

✨ SAMPLE RESULTS:
  • Nominees for the Digital Spy Reader Awards related to Casualty (Col: Nominee)
  • Future stadiums in England (Col: Stadium)
  • Non-elected members of the National Assembly of Zambia (1969–1973) (Col: Member)
  • Highest-grossing animated feature films released in 2022 (Col: Title)
  • Members of the 4th Lok Sabha nominated to the Indian Parliament (Col: Member)

──────────────────────────────────────────────────────────────────────────────────────────
                                   🕵️ REJECTION SAMPLES                                   
──

In [244]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(group_constructor.valid_tables)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/77 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/77 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     77
❌ Groups Discarded:  0
📈 Success Rate:      100.0%
-------------------------------------------------------
Strategy Distribution (Top 4 per table):
  - UNIQUE_DIVERSITY          : 62
  - COUNT_TOTAL               : 37
  - FREQUENCY_MODE            : 34
  - EXTREME_TEMPORAL          : 34
  - CATEGORICAL_FILTER_LOOKUP : 26
  - FILTER_COUNT_TEMPORAL     : 23
  - EXTREME_NUMERICAL         : 9
  - TOP_K_RANKING             : 8
  - AGGREGATION               : 5
  - FILTER_COUNT_NUMERICAL    : 3
  - COMPARATIVE_DIFFERENCE    : 1
-------------------------------------------------------
POSSIBLE STRATEGIES  | GROUP COUNT  | PERCENTAGE
-------------------------------------------------------
#1                   | 8            |      10.4%
#2                   | 16           |      20.8%
#3             

In [245]:
import random
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

generator = QuestionGenerator(selector.updated_data,
                              strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await generator.generate_questions()

🚀 Starting Question Generation for 77 tables...
--- Round 1: Processing 77 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/77 [00:00<?, ?it/s]

--- Round 2: Processing 2 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/2 [00:00<?, ?it/s]


═════════════════════════════════════════════════════════════════
                  📊 QUESTION GENERATION SUMMARY                  
═════════════════════════════════════════════════════════════════
Initial Tables:      77
Successfully Gen:    77
Exhausted Strats:    0
Success Rate:        100.0%
-----------------------------------------------------------------
STRATEGY                     | SUCCESS  | TRIED    | RATE
-----------------------------------------------------------------
UNIQUE_DIVERSITY             | 13       | 14       |  93%
COUNT_TOTAL                  | 12       | 12       | 100%
FILTER_COUNT_TEMPORAL        | 11       | 11       | 100%
CATEGORICAL_FILTER_LOOKUP    | 11       | 11       | 100%
EXTREME_TEMPORAL             | 11       | 11       | 100%
FREQUENCY_MODE               | 10       | 11       |  91%
AGGREGATION                  | 2        | 2        | 100%
EXTREME_NUMERICAL            | 2        | 2        | 100%
TOP_K_RANKING                | 2        | 2     

,Name (English),Name (Chinese)
0,Wang Xingda,王行达
1,Zhao Xidi,赵希迪
2,Wu Xiaoqiu,吴筱秋
3,Yu Zhenqi,于振起
4,Wu Hongbin,吴虹滨
5,Lu Guicheng,鲁桂成
6,Gong Jianwei,宫建伟
7,Cui Qiming,崔启明
8,Xie Xiaoyong,谢小用


-----------------------------------------------------------------

[2] Table: Diplomatic missions of Thailand in the Americas
    Original Data: https://en.wikipedia.org/wiki/List_of_diplomatic_missions_of_Thailand
    Strategy Attempted: FREQUENCY_MODE
    Rejection Reason:   There is a tie for the most frequent value in the 'Mission' column, making it impossible to identify a single absolute mode.
    Table Data:


,Mission,Host country
0,Embassy,Argentina
1,Embassy,Brazil
2,Embassy,Canada
3,Consulate-General,Canada
4,Embassy,Chile
5,Embassy,Mexico
6,Embassy,Peru
7,Embassy,United States
8,Consulate-General,United States
9,Consulate-General,United States


-----------------------------------------------------------------


In [246]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(generator.valid_results, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/66 [00:00<?, ?it/s]

Name
Catholic High School
Kormilda College
Darwin Adventist School
Our Lady of the Sacred Heart College
Our Lady of the Sacred Heart School
Our Lady of Victories School
Saint Michael's School


Name (English),
Wang Xingda,None
Zhao Xidi,None
Wu Xiaoqiu,None
Yu Zhenqi,None
Wu Hongbin,None
Lu Guicheng,None
Gong Jianwei,None
Cui Qiming,None
Xie Xiaoyong,None


Name (real name),Years active
Rick Riley,1978
Todd Swalla,1982 1983
Brian Damage (Brian Keats),1983
Myke Hideous (Myke Itzazone),1998
Zoli Téglás,2000
Renfield (Ken Schalk),2000
Matt Cross,2000


President,Residence
John Tyler,"Executive Mansion ( Richmond, Virginia )"
Grover Cleveland,"New York State Executive Mansion ( Albany , New York )"
Theodore Roosevelt,"New York State Executive Mansion ( Albany , New York )"
William Howard Taft,"Malacañang Palace ( Manila , Philippines)"
Woodrow Wilson,"Prospect House , Princeton, New Jersey"
Franklin D. Roosevelt,"New York State Executive Mansion ( Albany , New York )"
Jimmy Carter,"Georgia Governor's Mansion ( Atlanta, Georgia )"
George H. W. Bush,Residence of the United States Ambassador to the United Nations ( New York City )
George H. W. Bush,"Number One Observatory Circle ( Washington, D.C. )"
Bill Clinton,"Arkansas Governor's Mansion ( Little Rock, Arkansas )"


Name,Current Location
Portrait of Madeline Scott,Manchester Art Gallery
Platt Lane,"Tate Britain , London"
Portrait of Charles Rowley (1839-1933),Manchester Art Gallery
Spinning,Royal Jubilee Exhibition
Weaving,Royal Jubilee Exhibition
Commerce,Royal Jubilee Exhibition
Shipping,Royal Jubilee Exhibition
Corn,Royal Jubilee Exhibition
Wool,Royal Jubilee Exhibition
Iron,Royal Jubilee Exhibition


Name
Finnish Museum of Natural History
National Museum of Finland
Finnish National Gallery
Ateneum
Kiasma
Sinebrychoff Art Museum


Governor,Vote
"José Méndez Quijada , Democratic Action",48.00
"José Méndez Quijada , Democratic Action",48.46
"Luis González , Democratic Action",39.07
"Luis González , Democratic Action",46.31
"Gilmer Viloria , Fifth Republic Movement",56.91
"Gilmer Viloria , Fifth Republic Movement",54.31
"Hugo Cabezas , United Socialist Party of Venezuela",59.47
"Henry Rangel Silva , United Socialist Party of Venezuela",82.30
"Henry Rangel Silva , PSUV",5975.00
"Gerardo Marquez , PSUV",4167.00


In [247]:
from knowledge_maze.create_dataset.create_questions.question_validator import ManualTagger

tagger = ManualTagger(validator.approved_data, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
tagger.display() # Render the output area

📦 Progress Loaded: 2 samples already tagged in 'tagger_progress.json'.


Output()

In [290]:
from knowledge_maze.create_dataset.create_questions.visualization import GroupMaker
import numpy as np

# gm = GroupMaker(debug=True)
gm = GroupMaker()
res_stats = []
guess = []
aaaaaaa = False

list_of_cleaned_data = [a for a in ttttt_cleaned_data if a['table_name']['header_chain'].lower().startswith('list of')]

for sample in random.sample(list_of_cleaned_data, 60):
    ctx = sample['table_name']
    data = sample['rows']

    print(f"Context: {ctx['header_chain']}")
    # print(f"Data:")
    # for row in data[:3]:
    #     print(row)
    # display(pd.DataFrame(data))
    # print(f"Debug process:")
    ent_col = gm.identify_entity_column(ctx, data)
    if ent_col:
        res_stats.append(True)
    else:
        res_stats.append(False)
    print(f"Identified Entity Column: {ent_col}")
    for idx, c in enumerate(data[0].keys()):
        if ent_col == c:
            guess.append(idx)
            aaaaaaa = True
            break
    if not aaaaaaa:
        guess.append(100)
    else:
        aaaaaaa = False
    print("-"*100)

print(Counter(guess))
print(f"Summary: {100 * np.mean(res_stats):.1f}% of table are considered valid by this code.")

Context: List of newspapers in Chile > Print newspapers > Daily circulation > National circulation
Identified Entity Column: Newspaper
----------------------------------------------------------------------------------------------------
Context: List of 1987 box office number-one films in the United States > Highest-grossing films > In-Year Release
Identified Entity Column: Title
----------------------------------------------------------------------------------------------------
Context: List of awards and nominations received by Courtney Love > Film > Critics associations
Identified Entity Column: Association
----------------------------------------------------------------------------------------------------
Context: List of Atlanta Braves team records > Table key
Identified Entity Column: None
----------------------------------------------------------------------------------------------------
Context: List of residences of presidents of the United States > Other official residences oc